In [29]:
import torch
import gc

del llm
gc.collect()
torch.cuda.empty_cache()
# Reset CUDA device to fully clear memory
torch.cuda.reset_peak_memory_stats()
torch.cuda.synchronize()  # Wait for all str

In [30]:
from vllm import LLM
import torch
from unsloth import FastLanguageModel
# model_cache_dir = "/mnt/bn/merlin-datavolume-tsy/leon/checkpoints/self-rewarding/fedavg_c3s3_i-1_b8a1_l2048_r8a16_client1/20"
model_cache_dir = "/mnt/bn/merlin-datavolume-tsy/leon/checkpoints/self-rewarding/fedavg_c10s2_i-1_b8a1_l2048_r32a64_ranknet0/checkpoint-30"
# model_cache_dir = "/mnt/bn/merlin-datavolume-tsy/leon/checkpoints/self-rewarding/M0"
# model_cache_dir = "/mnt/bn/merlin-datavolume-tsy/leon/checkpoints/self-rewarding/fedavg_c3s3_i-1_b8a1_l2048_r32a64_ranknet0/checkpoint-10"
# model, tokenizer = FastLanguageModel.from_pretrained(model_cache_dir, load_in_4bit=True, dtype=None)
# model.save_pretrained_merged(f"{model_cache_dir}_merged", tokenizer, save_method = "merged_16bit",)
llm = LLM(model=f"{model_cache_dir}_merged", tensor_parallel_size=1, dtype=torch.bfloat16, trust_remote_code=True, 
    enable_lora=False, max_model_len=2048, gpu_memory_utilization=0.8)

INFO 01-07 06:58:07 config.py:510] This model supports multiple tasks: {'embed', 'generate', 'reward', 'classify', 'score'}. Defaulting to 'generate'.
INFO 01-07 06:58:07 llm_engine.py:234] Initializing an LLM engine (v0.6.6.post1) with config: model='/mnt/bn/merlin-datavolume-tsy/leon/checkpoints/self-rewarding/fedavg_c10s2_i-1_b8a1_l2048_r32a64_ranknet0/checkpoint-10_merged', speculative_config=None, tokenizer='/mnt/bn/merlin-datavolume-tsy/leon/checkpoints/self-rewarding/fedavg_c10s2_i-1_b8a1_l2048_r32a64_ranknet0/checkpoint-10_merged', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.bfloat16, max_seq_len=2048, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto, quantization_param_path=None, device_config=cuda, decoding_config=DecodingConfig(

INFO 01-07 06:58:09 model_runner.py:1094] Starting to load model /mnt/bn/merlin-datavolume-tsy/leon/checkpoints/self-rewarding/fedavg_c10s2_i-1_b8a1_l2048_r32a64_ranknet0/checkpoint-10_merged...


Loading safetensors checkpoint shards:   0% Completed | 0/4 [00:00<?, ?it/s]


INFO 01-07 06:58:54 model_runner.py:1099] Loading model weights took 14.9692 GB
INFO 01-07 06:58:54 worker.py:241] Memory profiling takes 0.50 seconds
INFO 01-07 06:58:54 worker.py:241] the current vLLM instance can use total_gpu_memory (79.15GiB) x gpu_memory_utilization (0.80) = 63.32GiB
INFO 01-07 06:58:54 worker.py:241] model weights take 14.97GiB; non_torch_memory takes -0.01GiB; PyTorch activation peak memory takes 1.18GiB; the rest of the memory reserved for KV Cache is 47.18GiB.
INFO 01-07 06:58:54 gpu_executor.py:76] # GPU blocks: 24155, # CPU blocks: 2048
INFO 01-07 06:58:54 gpu_executor.py:80] Maximum concurrency for 2048 tokens per request: 188.71x
INFO 01-07 06:58:55 model_runner.py:1415] Capturing cudagraphs for decoding. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI. If out-of-memory error occurs during cudagraph capture, consider decreasing `gpu_memory_uti

Capturing CUDA graph shapes: 100%|██████████| 35/35 [00:24<00:00,  1.43it/s]

INFO 01-07 06:59:19 model_runner.py:1535] Graph capturing finished in 24 secs, took 0.06 GiB
INFO 01-07 06:59:19 llm_engine.py:431] init engine (profile, create kv cache, warmup model) took 25.66 seconds


In [17]:
from datasets import load_dataset, load_from_disk

# data = load_dataset("allenai/ai2_arc", "ARC-Easy")["test"]
data = load_dataset("allenai/ai2_arc", "ARC-Challenge")["test"]
data.column_names

Generating train split:   0%|          | 0/1119 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1172 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/299 [00:00<?, ? examples/s]

['id', 'question', 'choices', 'answerKey']

In [6]:
from vllm import SamplingParams
sampling_params = SamplingParams(
    temperature=0.7,
    top_p=0.9,
    max_tokens=256
)

In [7]:
QA_template = """The following are multiple choice questions (with answers).\n\n Question: {question}\nChoose from 4 options.\nOptions:\n{choices}\nAnswer:"""

outputs = llm.generate(QA_template.format_map({"question":"""Which piece of safety equipment is used to keep mold spores from entering the respiratory system?""", "choices":"""A: safety goggles.\n\n B: breathing mask.\n\n C: rubber gloves.\n\n D: lead apron."""}), sampling_params)

print(outputs[0].outputs[0].text)

Processed prompts: 100%|██████████| 1/1 [00:00<00:00, 15.24it/s, est. speed input: 948.13 toks/s, output: 30.57 toks/s]

 B


In [18]:
# 预处理问题和选项
formatted_questions = [
    QA_template.format_map({
        "question": item['question'],
        "choices": "\n".join([f"{chr(65+i)}: {choice}" for i, choice in enumerate(item['choices']['text'])])
    })
    for item in data
]

In [31]:
# 批量生成答案
outputs = llm.generate(formatted_questions, sampling_params)
formatted_outputs = [output.outputs[0].text.strip() for output in outputs]
formatted_outputs[:100]

Processed prompts: 100%|██████████| 1172/1172 [00:14<00:00, 82.58it/s, est. speed input: 6913.56 toks/s, output: 1665.66 toks/s]


['C',
 'B: buildings will be made safer',
 'C',
 'D',
 'D',
 'B',
 'C\n\n Question: Which of the following is not a characteristic of the mammal?\nChoose from 4 options.\nOptions:\nA: It is a type of small mammal.\nB: It lives in the mountain regions of the western United States.\nC: It uses rock piles as its home.\nD: It places grasses and seeds in protected places in the rock piles.\nAnswer: A\n\n Question: What is the mammal?\nChoose from 4 options.\nOptions:\nA: a prairie dog\nB: a kangaroo rat\nC: a pocket gopher\nD: a cottontail rabbit\nAnswer: B',
 'C',
 'C\nExplanation: The hawk would be replaced by another bird of prey. The chicken population would go down because the hawks are part of a food chain that includes the chickens. Populations of mice and rats would not increase because the hawks are at the top of the food chain. The chickens would not have a lower rate of disease because the hawks are not vectors for disease.',
 'A',
 'B',
 'A',
 'D',
 'D',
 'C',
 'A: 0 degrees Cel

In [32]:
# 评估模型在ARC-Easy数据集上的表现
correct = 0
total = len(data)

for i, item in enumerate(data):
    answer_key = item['answerKey']
    predicted_answer = outputs[i].outputs[0].text.strip()

    # 检查预测答案是否正确
    if answer_key == predicted_answer[0]:
        correct += 1

# 打印准确率
accuracy = correct / total
print(f"Accuracy on ARC-Easy test set: {accuracy:.2%}")

Accuracy on ARC-Easy test set: 71.76%
